In [13]:
# Este comando descarga el repositorio entero a una carpeta llamada 'TFMDS' en Colab.
#!git clone https://github.com/jmorala/TFMDS.git

# Inicializar directorios
Clonar repositorio github
Posicionarse en el directorio raíz

In [14]:
import os

# Detectar si estamos en Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Configurar el directorio de trabajo según el entorno
if IN_COLAB:
    os.chdir('TFMDS')
else:
    # Detectar si estamos en Codespaces o VS Code local
    if os.path.exists('/workspaces/TFMDS'):
        # Entorno Codespaces
        os.chdir('/workspaces/TFMDS')
    else:
        # En VS Code local, nos movemos al directorio raíz del proyecto
        # Usa raw string para evitar errores de escape en rutas Windows
        current_dir = r'C:\Users\jmora\Documents\TFMDS'
        os.chdir(current_dir)

# OPCIONAL: Para verificar que estás en la ruta correcta y ver las carpetas
print("Directorio de trabajo actual:", os.getcwd())

Directorio de trabajo actual: C:\Users\jmora\Documents\TFMDS


## Lectura de fichero y adaptación de los tipos


In [15]:
import pandas as pd

# Ruta relativa del archivo CSV
RUTA_DATOS_TRAIN = 'datos/df_train.csv'

# Cargar el archivo en un DataFrame de Pandas
dfSTventasTrain = pd.read_csv(RUTA_DATOS_TRAIN, sep=';',
    parse_dates=['idSecuencia'])

# Cargar el conjunto de datos de prueba
RUTA_DATOS_TEST = 'datos/df_test.csv'
dfSTventasTest = pd.read_csv(RUTA_DATOS_TEST, sep=';',
    parse_dates=['idSecuencia'])


# Generar dataframes para distintos tests

In [16]:
# Filtrar filas donde blOpen == 0 (tienda cerrada) - no incluir en predicciones
dfSTventasTrain = dfSTventasTrain[dfSTventasTrain['bolOpen'] == 1].reset_index(drop=True)
dfSTventasTest = dfSTventasTest[dfSTventasTest['bolOpen'] == 1].reset_index(drop=True)

print(f"✅ Filas eliminadas donde blOpen == 0 (tienda cerrada)")
print(f"   dfSTventasTrain: {len(dfSTventasTrain)} registros")
print(f"   dfSTventasTest: {len(dfSTventasTest)} registros")

✅ Filas eliminadas donde blOpen == 0 (tienda cerrada)
   dfSTventasTrain: 521202 registros
   dfSTventasTest: 23244 registros


# Naive

In [17]:
# ============================================================================
# MÉTODO NAIVE: Predicciones basadas en el último valor observado del train
# ============================================================================

from lib.metricas import calcular_metricas, resumen_metricas, resumen_final_modelos
from lib.utils import obtener_top_productos_por_cluster

metricas_naive = []  # acumulador de resultados

# ---------------------------------------------------------------
# PASO 0: Calcular predicciones NAIVE a nivel de PRODUCTO INDIVIDUAL
# ---------------------------------------------------------------
# Para cada producto, usar el último valor del train como predicción para todos los días del test
df_test_naive = dfSTventasTest.copy()
df_train_ultimo = dfSTventasTrain.sort_values('idSecuencia').groupby('producto')['udsVenta'].last().reset_index()
df_train_ultimo.rename(columns={'udsVenta': 'ultimo_valor'}, inplace=True)

# Merge para obtener el último valor por producto
df_test_naive = df_test_naive.merge(df_train_ultimo[['producto', 'ultimo_valor']], on='producto', how='left')
df_test_naive['prediccion_naive'] = df_test_naive['ultimo_valor']

# ---------------------------------------------------------------
# 1. NAIVE SOBRE VENTAS DIARIAS TOTALES (GLOBAL)
# Agregar predicciones y reales por día
# ---------------------------------------------------------------
df_test_daily = df_test_naive.groupby('idSecuencia').agg({
    'udsVenta': 'sum',
    'prediccion_naive': 'sum'
}).reset_index()

met_total = calcular_metricas(
    y=df_test_daily['udsVenta'],
    y_pred=df_test_daily['prediccion_naive'],
    algoritmo='NAIVE', ndetalle='Global', cluster=None, producto=None)

metricas_naive.append(met_total)

# ---------------------------------------------------------------
# 2. NAIVE POR CLUSTER
# Agregar predicciones y reales por día y cluster
# ---------------------------------------------------------------
clusters = sorted(df_test_naive['Cluster'].dropna().unique())
for cl in clusters:
    df_test_cl = df_test_naive[df_test_naive['Cluster'] == cl]
    
    if df_test_cl.empty:
        print(f"⚠️ Cluster {cl} sin datos en test. Se omite.")
        continue
    
    df_test_cl_daily = df_test_cl.groupby('idSecuencia').agg({
        'udsVenta': 'sum',
        'prediccion_naive': 'sum'
    }).reset_index()
    
    met_cl = calcular_metricas(
        y=df_test_cl_daily['udsVenta'],
        y_pred=df_test_cl_daily['prediccion_naive'],
        algoritmo='NAIVE', ndetalle='Global',
        cluster=cl, producto=None,
    )
    metricas_naive.append(met_cl)
    
# ---------------------------------------------------------------
# 3. NAIVE POR PRODUCTOS TOP (2 productos por cluster según ventas totales en train)
# ---------------------------------------------------------------

top_productos = obtener_top_productos_por_cluster(
    df=dfSTventasTrain,
    col_ventas='udsVenta',
    col_cluster='Cluster',
    col_producto='producto',
    n_productos=2
)

for cluster, productos in top_productos.items():
    for producto in productos:
        # Filtrar datos de test para este producto y cluster
        df_test_prod = df_test_naive[(df_test_naive['Cluster'] == cluster) & (df_test_naive['producto'] == producto)]
        
        if df_test_prod.empty:
            print(f"⚠️ Producto {producto} en Cluster {cluster} sin datos en test. Se omite.")
            continue
        
        # Agrupar por día
        df_test_prod_daily = df_test_prod.groupby('idSecuencia').agg({
            'udsVenta': 'sum',
            'prediccion_naive': 'sum'
        }).reset_index()
        
        # Calcular métricas
        metricas_producto = calcular_metricas(
            y=df_test_prod_daily['udsVenta'],
            y_pred=df_test_prod_daily['prediccion_naive'],
            algoritmo='NAIVE',
            ndetalle='Global', cluster=cluster, producto=producto)
        metricas_naive.append(metricas_producto)


resumen_metricas(metricas_naive)


📊 RESUMEN DE MÉTRICAS
Algoritmo NDetalle  Cluster  Producto      MAE     RMSE      R2  MAPE (%)  SMAPE (%)  RMSSE  MAE (%)
    NAIVE   Global      3.0     314.0   0.0000   0.0000  1.0000       NaN       0.00    NaN      NaN
    NAIVE   Global      3.0     257.0   0.5000   1.2558 -0.1884    100.00     200.00 0.8544   100.00
    NAIVE   Global      0.0     413.0   1.6154   2.2014 -0.0012     23.93      98.66 0.7706    77.78
    NAIVE   Global      1.0      41.0   3.3846   4.5319 -0.0073     72.57      57.62 0.7736    51.16
    NAIVE   Global      2.0       2.0   3.6154   4.5742 -0.0234     75.79      65.31 0.6355    57.32
    NAIVE   Global      1.0     144.0   3.9615   5.1103 -0.0166     67.27      85.17 0.7638    70.07
    NAIVE   Global      0.0     294.0   7.3846   7.6912 -6.4162    228.63     150.61 1.7719   400.00
    NAIVE   Global      2.0       1.0   7.3462  12.1323 -0.1863     71.59      66.46 0.7638    62.21
    NAIVE   Global      2.0       NaN  20.5000  28.3868 -0.0836    1

# Media de 7 días

In [18]:
# ============================================================================
# MÉTODO MEDIA 7 DÍAS: Predicción = media de los últimos 7 días del train
# ============================================================================

from lib.metricas import calcular_metricas, resumen_metricas
import pandas as pd
import numpy as np

WINDOW = 7
metricas_media7 = []

print("\n" + "="*100)
print(f"🧮 MEDIA MÓVIL BASE ({WINDOW} días) - Baseline")
print("="*100)

# ---------------------------------------------------------------
# PASO 0: Calcular predicciones MEDIA 7 a nivel de PRODUCTO INDIVIDUAL
# ---------------------------------------------------------------
# Para cada producto, usar la media de los últimos 7 días del train como predicción
df_test_media7 = dfSTventasTest.copy()
df_train_sorted = dfSTventasTrain.sort_values('idSecuencia')
df_train_media7 = df_train_sorted.groupby('producto', group_keys=False).apply(lambda x: x.tail(WINDOW)).reset_index(drop=True)
df_train_media7_per_product = df_train_media7.groupby('producto')['udsVenta'].mean().reset_index()
df_train_media7_per_product.rename(columns={'udsVenta': 'media7_valor'}, inplace=True)

# Merge para obtener la media 7 días por producto
df_test_media7 = df_test_media7.merge(df_train_media7_per_product[['producto', 'media7_valor']], on='producto', how='left')
df_test_media7['prediccion_media7'] = df_test_media7['media7_valor']

# ---------------------------------------------------------------
# 1. MEDIA 7 DÍAS SOBRE VENTAS DIARIAS TOTALES (GLOBAL)
# Agregar predicciones y reales por día
# ---------------------------------------------------------------
df_test_daily = df_test_media7.groupby('idSecuencia').agg({
    'udsVenta': 'sum',
    'prediccion_media7': 'sum'
}).reset_index()

met_total = calcular_metricas(
    y=df_test_daily['udsVenta'],
    y_pred=df_test_daily['prediccion_media7'],
    algoritmo='MEDIA7', ndetalle='Global', cluster=None, producto=None)

metricas_media7.append(met_total)

# ---------------------------------------------------------------
# 2. MEDIA 7 DÍAS POR CLUSTER
# Agregar predicciones y reales por día y cluster
# ---------------------------------------------------------------
clusters = sorted(df_test_media7['Cluster'].dropna().unique())
for cl in clusters:
    df_test_cl = df_test_media7[df_test_media7['Cluster'] == cl]
    
    if df_test_cl.empty:
        print(f"⚠️ Cluster {cl} sin datos en test. Se omite.")
        continue
    
    df_test_cl_daily = df_test_cl.groupby('idSecuencia').agg({
        'udsVenta': 'sum',
        'prediccion_media7': 'sum'
    }).reset_index()
    
    met_cl = calcular_metricas(
        y=df_test_cl_daily['udsVenta'],
        y_pred=df_test_cl_daily['prediccion_media7'],
        algoritmo='MEDIA7', ndetalle='Global',
        cluster=cl, producto=None,
    )
    metricas_media7.append(met_cl)

# ---------------------------------------------------------------
# 3. MEDIA 7 DÍAS POR PRODUCTOS TOP (2 productos por cluster según ventas totales en train)
# ---------------------------------------------------------------

top_productos = obtener_top_productos_por_cluster(
    df=dfSTventasTrain,
    col_ventas='udsVenta',
    col_cluster='Cluster',
    col_producto='producto',
    n_productos=2
)

for cluster, productos in top_productos.items():
    for producto in productos:
        # Filtrar datos de test para este producto y cluster
        df_test_prod = df_test_media7[(df_test_media7['Cluster'] == cluster) & (df_test_media7['producto'] == producto)]
        
        if df_test_prod.empty:
            print(f"⚠️ Producto {producto} en Cluster {cluster} sin datos en test. Se omite.")
            continue
        
        # Agrupar por día
        df_test_prod_daily = df_test_prod.groupby('idSecuencia').agg({
            'udsVenta': 'sum',
            'prediccion_media7': 'sum'
        }).reset_index()
        
        # Calcular métricas
        metricas_producto = calcular_metricas(
            y=df_test_prod_daily['udsVenta'],
            y_pred=df_test_prod_daily['prediccion_media7'],
            algoritmo='MEDIA7', ndetalle='Global', cluster=cluster, producto=producto)
        metricas_media7.append(metricas_producto)

resumen_metricas(metricas_media7)

# ============================================================================
# CONSOLIDAR Y GUARDAR TODAS LAS MÉTRICAS BASELINE
# ============================================================================

# Combinar métricas de NAIVE y MEDIA7
todas_metricas_baseline = metricas_naive + metricas_media7

# Guardar resultados consolidados
df_resultados_baseline = pd.DataFrame(todas_metricas_baseline)
output_path_baseline = 'datos/resultados_metricas_baseline.csv'
df_resultados_baseline.to_csv(output_path_baseline, index=False)
print(f"\n💾 Todas las métricas Baseline guardadas en: {output_path_baseline}")
print(f"   Total de registros: {len(todas_metricas_baseline)} (NAIVE + MEDIA7)")
print("="*100)


🧮 MEDIA MÓVIL BASE (7 días) - Baseline

📊 RESUMEN DE MÉTRICAS
Algoritmo NDetalle  Cluster  Producto      MAE     RMSE      R2  MAPE (%)  SMAPE (%)  RMSSE  MAE (%)
   MEDIA7   Global      3.0     314.0   0.0000   0.0000  1.0000       NaN       0.00    NaN      NaN
   MEDIA7   Global      3.0     257.0   0.5000   1.2558 -0.1884    100.00     200.00 0.8544   100.00
   MEDIA7   Global      0.0     413.0   1.8462   2.2280 -0.0255     34.41     102.48 0.7799    88.89
   MEDIA7   Global      0.0     294.0   2.0000   2.8284 -0.0030     27.90     127.35 0.6516   108.33
   MEDIA7   Global      2.0       2.0   3.8901   4.7352 -0.0968     86.62      67.42 0.6579    61.67
   MEDIA7   Global      1.0     144.0   4.4670   5.3701 -0.1226     99.97      84.34 0.8027    79.01
   MEDIA7   Global      1.0      41.0   4.3297   5.6105 -0.5437     57.19      82.94 0.9577    65.45
   MEDIA7   Global      2.0       1.0   7.6209  12.4326 -0.2457     68.78      70.43 0.7827    64.54
   MEDIA7   Global      2.0 

C:\Users\jmora\AppData\Local\Temp\ipykernel_17996\957561924.py:22: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_train_media7 = df_train_sorted.groupby('producto', group_keys=False).apply(lambda x: x.tail(WINDOW)).reset_index(drop=True)
